# LLM Criticism and Human Correction

This notebook audits the Claude critic stage and the resulting human corrections. Execution, validation, append-only batch synchronization, and workbook creation are handled by `run_claude_criticism.py` following `claude_criticism_runbook.md`. Claude suggestions remain advisory: only rows explicitly reviewed by the human coder can replace Codex labels.


In [ ]:
from __future__ import annotations

from math import sqrt
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
LLM_DIR = CLASSIFICATION_DIR / "llm_annotation"
CORRECTION_DIR = CLASSIFICATION_DIR / "human_correction"

CLAUDE_DIR = LLM_DIR / "claude"
PILOT_METRICS_PATH = CLAUDE_DIR / "pilot/evaluation/critic_pilot_model_metrics.csv"
RUN_MANIFEST_PATH = CLAUDE_DIR / "run_manifest.csv"
PRODUCTION_MANIFEST_PATH = CLAUDE_DIR / "production/manifest.csv"
RANKED_REVIEW_PATH = CORRECTION_DIR / "frame_llm_ranked_review.xlsx"
AUDIT_PATH = CORRECTION_DIR / "frame_llm_residual_audit.xlsx"
FINAL_LABELS_PATH = CORRECTION_DIR / "frame_llm_correction_completed.csv"


## Pilot Calibration

The critic is calibrated only against the adjudicated human pilot and the matching high-reasoning Codex annotations. The selected production critic should produce valid hierarchical outputs, recover at least 60% of known Codex errors among its top 40 ranked pilot cases, and show no repeated systematic codebook blind spot. Opus is selected only if it recovers at least one additional top-40 error relative to Sonnet.


In [ ]:
if PILOT_METRICS_PATH.exists():
    pilot_metrics = pd.read_csv(PILOT_METRICS_PATH)
    display(pilot_metrics)
else:
    print("Pilot critic metrics are not available yet. Follow claude_criticism_runbook.md.")


## Production Coverage and Usage

Critic chunks preserve direct provenance from validated Codex annotation batches while limiting each Claude call to at most 25 rows. This check makes missing criticism, accidental batch changes, and future append-only extensions visible before human review.


In [ ]:
if PRODUCTION_MANIFEST_PATH.exists():
    critic_manifest = pd.read_csv(PRODUCTION_MANIFEST_PATH)
    print(f"Critic input batches: {len(critic_manifest):,}")
    print(f"Critic input rows: {critic_manifest['rows'].sum():,}")
    display(critic_manifest.tail())
else:
    print("Production critic inputs have not been synchronized yet.")

if RUN_MANIFEST_PATH.exists():
    run_manifest = pd.read_csv(RUN_MANIFEST_PATH)
    production_runs = run_manifest.loc[run_manifest['dataset'].eq('production')].copy()
    print(f"Recorded production attempts: {len(production_runs):,}")
    display(production_runs.groupby(['requested_model', 'effort', 'status'], dropna=False).size().rename('attempts').reset_index())
else:
    print("No Claude critic run metadata is available yet.")


## Human Review Yield

The ranked workbook contains the critic's highest-priority cases. Review proceeds in score order and can stop before the maximum only after two consecutive completed 100-case waves each produce fewer than five corrections. The separate random audit is generated after the ranked stopping point and estimates errors among otherwise-unreviewed rows.


In [ ]:
def parse_label(value: object) -> bool | None:
    if pd.isna(value) or str(value).strip().upper() == "NA":
        return None
    return str(value).strip().upper() == "TRUE"


def wilson_interval(successes: int, total: int, z: float = 1.96) -> tuple[float, float]:
    if total == 0:
        return (float("nan"), float("nan"))
    proportion = successes / total
    denominator = 1 + z**2 / total
    centre = (proportion + z**2 / (2 * total)) / denominator
    margin = z * sqrt(proportion * (1 - proportion) / total + z**2 / (4 * total**2)) / denominator
    return centre - margin, centre + margin


def review_summary(path: Path, source: str) -> pd.DataFrame:
    frame = pd.read_excel(path)
    reviewed = frame.loc[frame['review_status'].eq('reviewed')].copy()
    for axis in ['substantive_target_discourse', 'clinical_frame_present', 'lived_experience_frame_present']:
        reviewed[f'{axis}_changed'] = [
            parse_label(final) != parse_label(original)
            for final, original in zip(reviewed[f'final_{axis}'], reviewed[axis])
        ]
    reviewed['any_correction'] = reviewed[[f'{axis}_changed' for axis in ['substantive_target_discourse', 'clinical_frame_present', 'lived_experience_frame_present']]].any(axis=1)
    reviewed['review_source'] = source
    return reviewed

reviewed_parts = []
if RANKED_REVIEW_PATH.exists():
    ranked_reviewed = review_summary(RANKED_REVIEW_PATH, 'ranked')
    reviewed_parts.append(ranked_reviewed)
    ranked_reviewed['review_wave'] = ((ranked_reviewed['rank'] - 1) // 100) + 1
    display(ranked_reviewed.groupby('review_wave').agg(reviewed=('annotation_id', 'size'), corrections=('any_correction', 'sum'), mean_overall_error_prob=('overall_error_prob', 'mean')).reset_index())
else:
    print("Ranked review workbook is not available yet.")

if AUDIT_PATH.exists():
    audit_reviewed = review_summary(AUDIT_PATH, 'residual_audit')
    reviewed_parts.append(audit_reviewed)
    print(f"Residual audit reviewed: {len(audit_reviewed):,}")
    audit_corrections = int(audit_reviewed["any_correction"].sum())
    audit_low, audit_high = wilson_interval(audit_corrections, len(audit_reviewed))
    print(f"Residual audit corrections: {audit_corrections:,}")
    print(f"Residual audit error rate: {audit_corrections / len(audit_reviewed):.1%} (95% Wilson CI {audit_low:.1%}-{audit_high:.1%})")
else:
    print("Residual audit workbook is not available yet.")


## Final Corrected Training Labels

Finalization preserves Codex labels for every unreviewed row and applies only explicit human-reviewed changes. Codex confidence is retained because confidence is not used as a classifier feature or training weight.


In [ ]:
if FINAL_LABELS_PATH.exists():
    final_labels = pd.read_csv(FINAL_LABELS_PATH)
    print(f"Final corrected rows: {len(final_labels):,}")
    print(f"Human-reviewed rows: {(final_labels['correction_source'] == 'human_review').sum():,}")
    display(final_labels['corrected_derived_frame'].value_counts(dropna=False).rename_axis('frame').reset_index(name='rows'))
else:
    print("Final corrected labels are not available yet. Complete both review workbooks and run finalize-corrections.")
